In [1]:
import sys

sys.path.append("..")

from common.metrics import metrics
from common.preprocessing import preprocessing_rg3
from common.train_test_split import split_data
from common.model_composition import model_composition

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 데이터 불러오기
train_rg3 = pd.read_csv(r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_rg3.csv")

In [3]:
X_rg3, y_rg3 = preprocessing_rg3(train_rg3)

print(X_rg3.shape)
print(y_rg3.value_counts())

(1182, 23)
PassOrFail
0    1157
1      25
Name: count, dtype: int64


In [4]:
X_train_rg3, X_test_rg3, y_train_rg3, y_test_rg3 = split_data(
    X_rg3,
    y_rg3
)

print("Train:", X_train_rg3.shape)
print("Test :", X_test_rg3.shape)

print("\nTrain Target")
print(y_train_rg3.value_counts())

print("\nTest Target")
print(y_test_rg3.value_counts())

Train: (945, 23)
Test : (237, 23)

Train Target
PassOrFail
0    925
1     20
Name: count, dtype: int64

Test Target
PassOrFail
0    232
1      5
Name: count, dtype: int64


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

# 정상 / 불량 비율
neg = (y_train_rg3 == 0).sum()
pos = (y_train_rg3 == 1).sum()

weight = neg / pos

models = {
    "Logistic Regression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        class_weight="balanced",
        random_state=42
    ),

    "LightGBM": LGBMClassifier(
        class_weight="balanced",
        random_state=42,
        verbosity=-1
    ),

    "XGBoost": XGBClassifier(
        scale_pos_weight=weight,
        random_state=42
    ),

    "CatBoost": CatBoostClassifier(
        class_weights=[1, weight],
        random_state=42,
        verbose=0
    )
}


for model_name, model in models.items():

    print("\n" + "=" * 50)
    print(f"[{model_name} - Class Weight]")
    print("=" * 50)

    model.fit(X_train_rg3, y_train_rg3)

    y_pred_rg3 = model.predict(X_test_rg3)
    y_prob_rg3 = model.predict_proba(X_test_rg3)[:, 1]

    metrics(
        y_test_rg3,
        y_pred_rg3,
        y_prob_rg3
    )


[Logistic Regression - Class Weight]
Accuracy : 0.6877637130801688
Precision: 0.0273972602739726
Recall   : 0.4
F1       : 0.05128205128205128
PR-AUC   : 0.04721508939561678

Confusion Matrix


,Pred Normal,Pred Defect
Actual Normal,161,71
Actual Defect,3,2



[Random Forest - Class Weight]
Accuracy : 0.9535864978902954
Precision: 0.0
Recall   : 0.0
F1       : 0.0
PR-AUC   : 0.02230378056278636

Confusion Matrix


,Pred Normal,Pred Defect
Actual Normal,226,6
Actual Defect,5,0



[LightGBM - Class Weight]
Accuracy : 0.9535864978902954
Precision: 0.0
Recall   : 0.0
F1       : 0.0
PR-AUC   : 0.025448765739666538

Confusion Matrix


,Pred Normal,Pred Defect
Actual Normal,226,6
Actual Defect,5,0



[XGBoost - Class Weight]
Accuracy : 0.9535864978902954
Precision: 0.0
Recall   : 0.0
F1       : 0.0
PR-AUC   : 0.05028294394349983

Confusion Matrix


,Pred Normal,Pred Defect
Actual Normal,226,6
Actual Defect,5,0



[CatBoost - Class Weight]
Accuracy : 0.9535864978902954
Precision: 0.0
Recall   : 0.0
F1       : 0.0
PR-AUC   : 0.03175362293852427

Confusion Matrix


,Pred Normal,Pred Defect
Actual Normal,226,6
Actual Defect,5,0
